# 06 — Azure OpenAI Function Calling with MCP Tools

This notebook implements the full six-step lifecycle from notebook 01 by hand: convert MCP tools into an OpenAI function schema, let the model choose a tool, execute it, and get a final answer.

> Requires `AZURE_OPENAI_ENDPOINT` in your `.env` (see notebook 02) and the **Cognitive Services User** role on that resource.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from src.azure_openai_helper import apply_tool_result, chat_with_tools, get_azure_openai_client
from src.config import get_settings
from src.mcp_client import connect

settings = get_settings()
openai_client = get_azure_openai_client(settings)

In [ ]:
# Step 2: list_tools() -> convert to the OpenAI `tools=` schema.
async def get_tools_schema():
    async with connect(read_only=True) as client:
        return await client.tools_as_openai_functions(), client

# We keep a live connection open for the rest of this notebook so we don't
# reconnect on every cell.
from src.mcp_client import AzureMcpClient, build_server_params
mcp_client = AzureMcpClient(build_server_params(read_only=True))
await mcp_client.__aenter__()
tools_schema = await mcp_client.tools_as_openai_functions()
print(f"{len(tools_schema)} tools available to the model")

In [ ]:
# Step 3-4: send the prompt + tools to the model; it may return tool_calls
# instead of (or in addition to) plain text.
messages = [{"role": "user", "content": "List all of the resource groups in my subscription"}]
response = chat_with_tools(openai_client, settings.azure_openai_model, messages, tools_schema)
response_message = response.choices[0].message
messages.append(response_message)
print("Tool calls requested:", [c.function.name for c in (response_message.tool_calls or [])])

In [ ]:
# Step 5: execute every requested tool call and append the results.
import json

if response_message.tool_calls:
    for tool_call in response_message.tool_calls:
        args = json.loads(tool_call.function.arguments or "{}")
        result = await mcp_client.call_tool(tool_call.function.name, args)
        apply_tool_result(messages, tool_call, result.content)
else:
    print("Model answered directly without calling a tool.")

In [ ]:
# Step 6: ask the model for its final natural-language answer now that it has
# the tool results in context.
final_response = chat_with_tools(openai_client, settings.azure_openai_model, messages, tools_schema)
print(final_response.choices[0].message.content)

await mcp_client.__aexit__(None, None, None)

## Notes on model requirements & next steps

- Any tool-calling-capable Azure OpenAI deployment works (`gpt-4o`, `gpt-4o-mini`, etc.).
- Some models cap the number of tools you can pass in one request — if you hit that limit, scope namespaces down (see [`docs/07_server_modes_and_advanced_config.md`](../docs/07_server_modes_and_advanced_config.md)).

Continue to [`07_building_a_conversational_agent.ipynb`](07_building_a_conversational_agent.ipynb) to turn this single round trip into a full multi-turn agent.